## Advanced Rag System

In [3]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 1. Navigate to project root
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
os.chdir(project_root)
sys.path.insert(0, str(project_root))

load_dotenv()

# 2. Load centralized config & embedding model
from src.services.llm_services import load_config, get_text_embeddings

config = load_config("src/config/config.yaml")
embeddings = get_text_embeddings(config)

print(f"📂 Project Root: {os.getcwd()}")
print(f"✅ Embedding Model: {config['text_emb_provider']} / {config['text_emb_model']}")

c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:133: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📂 Project Root: c:\Users\User\Desktop\Operation Legder-Mind
✅ Embedding Model: sbert / sentence-transformers/all-MiniLM-L6-v2


In [6]:
import weaviate
from langchain_weaviate import WeaviateVectorStore

# 1. Connect to local Weaviate running in Docker
client = weaviate.connect_to_local()
INDEX_NAME = "FinancialReportChunks"

if not client.is_ready():
    raise ConnectionError("❌ Local Weaviate is not running. Start Docker container first!")

# 2. Connect to the existing collection (No re-indexing required!)
vectorstore = WeaviateVectorStore(
    client=client,
    index_name=INDEX_NAME,
    text_key="text",
    embedding=embeddings
)

# 3. Check collection count
collection = client.collections.get(INDEX_NAME)
count = collection.aggregate.over_all(total_count=True).total_count

print(f"✅ Connected to existing Weaviate collection '{INDEX_NAME}'")
print(f"📊 Total indexed chunks available: {count}")

✅ Connected to existing Weaviate collection 'FinancialReportChunks'
📊 Total indexed chunks available: 543


In [7]:
query = "What is Uber's core mission statement as described in the 2024 annual report?"

# Perform similarity search
docs = vectorstore.similarity_search(query, k=3)

print("=" * 60)
print(f"🔍 QUERY: {query}")
print("=" * 60)

for i, doc in enumerate(docs, 1):
    page = doc.metadata.get("page", "N/A")
    splitter = doc.metadata.get("splitter", "N/A")
    
    print(f"\n🎯 [Rank #{i}] (SEC 10-K Page: {page} | Strategy: {splitter})")
    print("-" * 60)
    print(doc.page_content.strip())
    print("-" * 60)

🔍 QUERY: What is Uber's core mission statement as described in the 2024 annual report?

🎯 [Rank #1] (SEC 10-K Page: 1.0 | Strategy: recursive)
------------------------------------------------------------
Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who are relentless about our  
mission to help people go anywhere and get anything and earn their way. 
Movement is what we power. It’s our lifeblood. It runs through our veins. It’s 
what gets us out of bed each morning. It pushes us to constantly reimagine 
how we can move better. For you. For all the places you want to go. For all the 
things you want to get. For all the ways you want to earn. Across the entire 
world. In real time. At the incredible speed of now.
------------------------------------------------------------

🎯 [Rank #2] (SEC 10-K Page: 11.0 | Strategy: recursive)
------------------------------------------------------------
Employees have access to an in

## Hybrid Search(Dense(vector search) +Sparce(BM25))

In [8]:
from collections import defaultdict
from langchain_core.documents import Document


# 1. Dense Search using Weaviate
def dense_search(query: str, top_k: int = 8):
    """Search using Dense Vectors (SBERT)."""
    query_vector = embeddings.embed_query(query)
    response = collection.query.near_vector(
        near_vector=query_vector,
        limit=top_k,
        return_metadata=["distance"]
    )
    
    results = []
    for obj in response.objects:
        doc = Document(
            page_content=obj.properties.get("text", ""),
            metadata={
                "page": obj.properties.get("page"),
                "source": obj.properties.get("source"),
                "splitter": obj.properties.get("splitter"),
            }
        )
        results.append({
            "doc": doc,
            "score": float(obj.metadata.distance) if obj.metadata else 0.0,
            "doc_id": str(obj.uuid)
        })
    return results


# 2. Sparse Search using Weaviate's Built-in BM25
def bm25_search(query: str, top_k: int = 8):
    """Search using Weaviate's native BM25 sparse keyword engine."""
    response = collection.query.bm25(
        query=query,
        limit=top_k,
        return_metadata=["score"]
    )
    
    results = []
    for obj in response.objects:
        doc = Document(
            page_content=obj.properties.get("text", ""),
            metadata={
                "page": obj.properties.get("page"),
                "source": obj.properties.get("source"),
                "splitter": obj.properties.get("splitter"),
            }
        )
        results.append({
            "doc": doc,
            "score": float(obj.metadata.score) if obj.metadata else 0.0,
            "doc_id": str(obj.uuid)
        })
    return results

# 3. Reciprocal Rank Fusion (RRF) matching your dictionary format
def rrf_fusion(dense_results, bm25_results, k: int = 60, top_k: int = 3):
    """
    Fuse dense and sparse results using RRF.
    Returns: List of dicts with 'doc', 'score', and 'doc_id'
    """
    rrf_scores = defaultdict(float)
    doc_lookup = {}
    # Rank Dense
    for rank, item in enumerate(dense_results, 1):
        doc_id = item["doc_id"]
        doc_lookup[doc_id] = item["doc"]
        rrf_scores[doc_id] += 1.0 / (k + rank)
    # Rank BM25
    for rank, item in enumerate(bm25_results, 1):
        doc_id = item["doc_id"]
        doc_lookup[doc_id] = item["doc"]
        rrf_scores[doc_id] += 1.0 / (k + rank)
    # Sort descending by RRF score
    sorted_items = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    fused = []
    for doc_id, score in sorted_items[:top_k]:
        fused.append({
            "doc": doc_lookup[doc_id],
            "score": score,
            "doc_id": doc_id
        })
    return fused

In [ ]:
query = "What is Uber's core mission statement as described in the 2024 annual report?"
# 1. Run Dense and BM25 searches
dense_results = dense_search(query, top_k=8)
bm25_results = bm25_search(query, top_k=8)
# 2. Test hybrid fusion
fused_results = rrf_fusion(dense_results, bm25_results, k=60, top_k=3)
# 3. Print results (your exact format)
print(f"Hybrid (RRF) search: '{query}'\n")
for i, res in enumerate(fused_results, 1):
    page = res['doc'].metadata.get('page', 'N/A')
    print(f"  [{i}] (RRF: {res['score']:.4f} | Page: {page}) {res['doc'].page_content[:120]}...")

🔀 Hybrid (RRF) search: 'What is Uber's core mission statement as described in the 2024 annual report?'

  [1] (RRF: 0.0328 | Page: 1.0) Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who a...
  [2] (RRF: 0.0161 | Page: 11.0) Employees have access to an internal jobs marketplace for full-time jobs as well as short-term stretch assignments that ...
  [3] (RRF: 0.0161 | Page: 141.0) f 2021 Proxy statement
1725 3rd Street
San Francisco, California 94158
uber.com...


## Reranking

In [13]:
from collections import defaultdict
from typing import Dict, Any, List
from sentence_transformers import CrossEncoder
from langchain_core.documents import Document
from src.services.llm_services import get_llm

# 1. Initialize Cross-Encoder Reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 2. Initialize LLM
try:
    llm = get_llm(config)
except Exception:
    llm = None


def rrf_fusion_scores(dense_results: List[Document], bm25_results: List[Document], k: int = 60) -> List[Dict[str, Any]]:
    """Reciprocal Rank Fusion (RRF): score(d) = sum(1 / (k + rank(d)))"""
    rrf_scores = defaultdict(float)
    doc_map = {}

    for rank, doc in enumerate(dense_results, 1):
        doc_id = doc.metadata.get("page_content_hash") or hash(doc.page_content)
        doc_map[doc_id] = doc
        rrf_scores[doc_id] += 1.0 / (k + rank)

    for rank, doc in enumerate(bm25_results, 1):
        doc_id = doc.metadata.get("page_content_hash") or hash(doc.page_content)
        doc_map[doc_id] = doc
        rrf_scores[doc_id] += 1.0 / (k + rank)

    sorted_items = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [{"doc": doc_map[doc_id], "rrf_score": score, "doc_id": doc_id} for doc_id, score in sorted_items]


def query_librarian(
    question: str,
    top_k_retrieval: int = 10,
    top_k_rerank: int = 3,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    'The Librarian' - Advanced Hybrid RAG Pipeline
    Constraint: query_librarian(question)
    """
    if not client.is_connected():
        client.connect()
    collection = client.collections.get("FinancialReportChunks")

    # 1. HYBRID RETRIEVAL (Dense + BM25)
    # A. Dense SBERT Search
    query_vector = embeddings.embed_query(question)
    dense_res = collection.query.near_vector(near_vector=query_vector, limit=top_k_retrieval, return_metadata=["distance"])
    dense_docs = [
        Document(
            page_content=obj.properties.get("text", ""),
            metadata={"page": obj.properties.get("page"), "source": obj.properties.get("source"), "splitter": obj.properties.get("splitter")}
        )
        for obj in dense_res.objects
    ]

    # B. Sparse BM25 Keyword Search
    bm25_res = collection.query.bm25(query=question, limit=top_k_retrieval, return_metadata=["score"])
    bm25_docs = [
        Document(
            page_content=obj.properties.get("text", ""),
            metadata={"page": obj.properties.get("page"), "source": obj.properties.get("source"), "splitter": obj.properties.get("splitter")}
        )
        for obj in bm25_res.objects
    ]

    # 2. REFINEMENT 1: Reciprocal Rank Fusion (RRF)
    rrf_candidates = rrf_fusion_scores(dense_docs, bm25_docs, k=60)[:top_k_retrieval]

    # 3. REFINEMENT 2: Cross-Encoder Neural Re-ranking
    candidate_docs = [item["doc"] for item in rrf_candidates]
    ce_pairs = [[question, doc.page_content] for doc in candidate_docs]
    ce_scores = reranker.predict(ce_pairs)

    reranked_pool = []
    for item, ce_score in zip(rrf_candidates, ce_scores):
        reranked_pool.append({
            "doc": item["doc"],
            "rrf_score": item["rrf_score"],
            "cross_encoder_score": float(ce_score),
            "page": item["doc"].metadata.get("page"),
        })

    reranked_pool.sort(key=lambda x: x["cross_encoder_score"], reverse=True)
    final_passages = reranked_pool[:top_k_rerank]

    # 4. GROUNDED FINANCIAL GENERATION
    context_blocks = [f"[SEC Form 10-K Page {p['page']}]:\n{p['doc'].page_content.strip()}" for p in final_passages]
    formatted_context = "\n\n".join(context_blocks)

    system_instruction = (
        "You are 'The Librarian', an authoritative financial analyst AI specializing in SEC Form 10-K filings. "
        "Answer the question accurately, concisely, and factually based strictly on the provided context. "
        "Always cite the relevant SEC 10-K page numbers for financial metrics and claims. "
        "If the answer cannot be determined directly from the context, state: 'Based on the provided document, this information is not available.'"
    )

    prompt = (
        f"{system_instruction}\n\n"
        f"--- FINANCIAL DOCUMENT EVIDENCE ---\n{formatted_context}\n\n"
        f"--- INQUIRY ---\n{question}\n\n"
        f"--- AUTHORITATIVE ANSWER ---"
    )

    if llm:
        try:
            response = llm.invoke(prompt)
            answer = response.content if hasattr(response, "content") else str(response)
        except Exception as e:
            answer = f"⚠️ LLM generation failed: {e}"
    else:
        answer = f"Evidence from SEC 10-K Page {final_passages[0]['page']}:\n{final_passages[0]['doc'].page_content.strip()[:300]}..."

    if verbose:
        print("=" * 70)
        print("📚 'THE LIBRARIAN' ADVANCED RAG PIPELINE")
        print("=" * 70)
        print(f"❓ Question: {question}")
        print(f"🔍 Hybrid Search:  {len(dense_docs)} Dense + {len(bm25_docs)} BM25 results")
        print(f"🔀 RRF Refinement: Merged into {len(rrf_candidates)} candidates")
        print(f"🎯 Cross-Encoder:  Filtered to Top {len(final_passages)} passages")
        print("-" * 70)
        print(f"💡 ANSWER:\n{answer.strip()}\n")
        print("-" * 70)
        print("📑 RE-RANKED CITATIONS:")
        for rank, p in enumerate(final_passages, 1):
            print(f"  [{rank}] Page {p['page']} | CE Score: {p['cross_encoder_score']:+.4f} | RRF Score: {p['rrf_score']:.5f}")
        print("=" * 70)

    return {
        "question": question,
        "answer": answer.strip(),
        "sources": [p["page"] for p in final_passages],
        "reranked_chunks": final_passages,
    }

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [14]:
# Test 1: Numerical entity & Gross Bookings
result_1 = query_librarian("What was Uber's total Gross Bookings reported for the fiscal year 2024?")
# Test 2: Specific corporate program entity
result_2 = query_librarian("What benefits do Uber One members receive according to the report?")

📚 'THE LIBRARIAN' ADVANCED RAG PIPELINE
❓ Question: What was Uber's total Gross Bookings reported for the fiscal year 2024?
🔍 Hybrid Search:  10 Dense + 10 BM25 results
🔀 RRF Refinement: Merged into 10 candidates
🎯 Cross-Encoder:  Filtered to Top 3 passages
----------------------------------------------------------------------
💡 ANSWER:
⚠️ LLM generation failed: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402, 'metadata': {'limit_source': 'openrouter_credits', 'remedy_hint': 'Add credits at https://openrouter.ai/settings/credits, or lower max_tokens / prompt size to fit your remaining balance.'}}}

----------------------------------------------------------------------
📑 RE-RANKED CITATIONS:
  [1] Page 63.0 | CE Score: +4.5517 | RRF Score: 0.03101
  [2] Page 57.0 | CE Score: +1.4412 | RRF Score: 0.01587

# Sanity Check (LLM as Judge & Rouge-L)

In [ ]:
import os
import json
import re
import time
from pathlib import Path
from dotenv import load_dotenv
import google.genai as genai
from rouge_score import rouge_scorer
from src.services.llm_services import get_llm

# Load API key from .env
load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

# Ensure query_librarian's generation LLM also uses Gemini (instead of OpenRouter)
if gemini_api_key:
    llm = get_llm(config, llm_provider="gemini", llm_model="gemini-3.8-flash")

if not gemini_api_key:
    print("⚠️ WARNING: No Gemini API key found in .env file")
    print("   Looking for GEMINI_API_KEY or GOOGLE_API_KEY")
    print("   LLM-as-Judge evaluation will be skipped")
    use_llm_judge = False
    judge_model = None
else:
    # Configure Gemini with new API (google.genai)
    # NOTE: Keep variable named judge_model (do not overwrite Weaviate 'client'!)
    judge_model = genai.Client(api_key=gemini_api_key)
    model_name = "gemini-3.8-flash"
    use_llm_judge = True
    print(f"✅ Gemini API configured with model: {model_name}")
    print("   Using google.genai package")

# Initialize ROUGE scorer
rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)


def evaluate_with_llm_judge(question, prediction, target, context=None):
    """
    Use Gemini as a judge to evaluate financial RAG prediction quality.
    Returns a score from 1-5 and reasoning.
    """
    if not use_llm_judge:
        return None, "API key not available"

    context_section = f"\nRETRIEVED CONTEXT:\n{context}\n" if context else ""

    judge_prompt = f"""You are an expert evaluator for financial analysis AI assistants specializing in SEC Form 10-K filings. Evaluate the following RAG output.
{context_section}
QUESTION: {question}

REFERENCE ANSWER: {target}

MODEL OUTPUT: {prediction}

Evaluate the MODEL OUTPUT based on:
1. **Accuracy & Numerical Precision**: Are financial figures (dollar amounts, percentages, dates) exact compared to the reference?
2. **Completeness**: Does it cover the key points from the reference?
3. **Grounding & Clarity**: Is it strictly grounded without hallucination and easy to understand?
4. **Faithfulness**: If the information is unavailable, does it appropriately refuse rather than guess?

Provide:
- A score from 1-5 (1=poor, 5=excellent)
- Brief reasoning (2-3 sentences)

Format your response EXACTLY as:
SCORE: [number]
REASONING: [your reasoning]"""

    try:
        # Use new google.genai API
        response = judge_model.models.generate_content(
            model=model_name,
            contents=judge_prompt
        )
        response_text = response.text.strip()

        # Parse score
        score_match = re.search(r"SCORE:\s*([1-5])", response_text, re.IGNORECASE)
        score = int(score_match.group(1)) if score_match else None

        # Parse reasoning
        reasoning_match = re.search(r"REASONING:\s*(.*)", response_text, re.IGNORECASE | re.DOTALL)
        reasoning = reasoning_match.group(1).strip() if reasoning_match else response_text

        return score, reasoning

    except Exception as e:
        print(f"⚠️ Gemini API error: {e}")
        return None, str(e)


# Load test dataset from artifacts/data/test.jsonl
test_path = Path("artifacts/data/test.jsonl")
with open(test_path, "r", encoding="utf-8") as f:
    test_dataset = [json.loads(line) for line in f if line.strip()]

# Evaluate on subset
eval_size = min(20, len(test_dataset))
eval_subset = test_dataset[:eval_size]

print(f"\n📊 Evaluating 'The Librarian' RAG on {eval_size} test examples...")
print("   Metrics: LLM-as-Judge (Gemini) + ROUGE-L\n")

llm_scores = []
rouge_l_scores = []
low_quality_cases = []

for idx, example in enumerate(eval_subset):
    print(f"Evaluating example {idx+1}/{eval_size}...", end="\r")

    question = example["question"]
    target = example["answer"]

    # Generate using Advanced Hybrid RAG Pipeline
    rag_out = query_librarian(question, verbose=False)
    prediction = rag_out["answer"]
    retrieved_context = "\n\n".join(
        f"[Page {p['page']}]: {p['doc'].page_content.strip()}"
        for p in rag_out["reranked_chunks"]
    )

    # Compute ROUGE-L
    rouge_result = rouge_scorer_obj.score(target, prediction)
    rouge_l_f1 = rouge_result['rougeL'].fmeasure
    rouge_l_scores.append(rouge_l_f1)

    # LLM as Judge (with rate limiting)
    if use_llm_judge:
        llm_score, reasoning = evaluate_with_llm_judge(
            question, prediction, target, context=retrieved_context
        )
        if llm_score is not None:
            llm_scores.append(llm_score)

        # Track low-quality cases (LLM score <= 2 or ROUGE-L < 0.3)
        if (llm_score and llm_score <= 2) or rouge_l_f1 < 0.3:
            if len(low_quality_cases) < 5:
                low_quality_cases.append({
                    "idx": idx,
                    "question": question,
                    "target": target,
                    "prediction": prediction,
                    "llm_score": llm_score,
                    "rouge_l": rouge_l_f1,
                    "reasoning": reasoning if llm_score else "N/A",
                })

        # Rate limiting for Gemini API (avoid hitting quota)
        time.sleep(1)

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"Number of samples: {eval_size}")

if use_llm_judge and llm_scores:
    avg_llm_score = sum(llm_scores) / len(llm_scores)
    print("\n📊 LLM-as-Judge (Gemini):")
    print(f"   Average Score: {avg_llm_score:.2f}/5.0")
    print(f"   Evaluated: {len(llm_scores)}/{eval_size} examples")

print("\n📊 ROUGE-L F1:")
print(f"   Average: {sum(rouge_l_scores)/len(rouge_l_scores):.4f}")

print("\n" + "-"*60)
print(f"LOW QUALITY CASES (showing {len(low_quality_cases)})")
print("-"*60)

for i, case in enumerate(low_quality_cases, 1):
    print(f"\n[{i}] Example {case['idx']}")
    if case['llm_score']:
        print(f"   LLM Score: {case['llm_score']}/5")
    print(f"   ROUGE-L: {case['rouge_l']:.3f}")
    print(f"   Question: {case['question'][:100]}...")
    print(f"   Target: {case['target'][:120]}...")
    print(f"   Prediction: {case['prediction'][:120]}...")
    if case['reasoning'] != "N/A":
        print(f"   Judge Reasoning: {case['reasoning'][:150]}...")
    print("-"*60)

print("\n" + "="*60)
print("ℹ️ Note: LLM-as-Judge provides nuanced evaluation but is subject to the judge model's own biases.")
print("   ROUGE-L measures lexical overlap and is more objective but may miss semantic similarity.")
print("="*60)